In [43]:
from collections import Counter
import pandas as pd


from icecube import icetray, dataio
from icecube import LeptonInjector


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


In [3]:
electron_weights = pd.read_csv("/project/def-nahee/kbas/Graphnet-Applications/Metadata/EventWeights/String340MC/Electron_LIW.csv")
muon_weights = pd.read_csv("/project/def-nahee/kbas/Graphnet-Applications/Metadata/EventWeights/String340MC/Muon_LIW.csv")
nc_weights = pd.read_csv("/project/def-nahee/kbas/Graphnet-Applications/Metadata/EventWeights/String340MC/NC_LIW.csv")
tau_weights = pd.read_csv("/project/def-nahee/kbas/Graphnet-Applications/Metadata/EventWeights/String340MC/Tau_LIW.csv")


In [4]:
electron_log = pd.read_csv("/home/kbas/scratch/String340MC/Logs/LIW/Electron_file_stats.csv")
muon_log = pd.read_csv("/home/kbas/scratch/String340MC/Logs/LIW/Muon_file_stats.csv")
tau_log = pd.read_csv("/home/kbas/scratch/String340MC/Logs/LIW/Tau_file_stats.csv")
nc_log = pd.read_csv("/home/kbas/scratch/String340MC/Logs/LIW/NC_file_stats.csv")


# Files that 0 event weight (probably because i3 file is not accesible or there is no daq)

all of these should have 
- status = not_ok
- weights_calculated = 0
- and worths to check the error message
- also, these any of these files should not have any data in flavor_weights

bunlarin disinda da 0 olabilie btw

## Electron

In [5]:
len(electron_log[electron_log["i3_file_opened_ok"] == False])


8

In [6]:
electron_log[electron_log["i3_file_opened_ok"] == False]


,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s
188,1080,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_1080.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1080.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.355621
805,1641,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_1641.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1641.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.291339
855,1687,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_1687.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1687.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.502012
2594,354,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_354.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_354.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.812702
3759,5235,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_5235.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_5235.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.116117
4543,6428,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_6428.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_6428.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.548010
4676,6629,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_6629.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_6629.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.248657
6631,9619,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_9619.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_9619.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.206379


#### first idea is to check if these files are completely empthy. 

In [7]:
bad_electrons = electron_log.loc[
    electron_log["i3_file_opened_ok"] == False,
    "i3_file"
]

In [8]:
# check if there is any kind of frame:

for f in bad_electrons:
    i3 = dataio.I3File(f)

    if not i3.more():
        print("EMPTY:", f)
        i3.close()
        continue

    frame = i3.pop_frame()
    print("READ OK:", f, frame.Stop)

    i3.close()


EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1080.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1641.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1687.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_354.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_5235.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_6428.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_6629.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_9619.i3.zst


#### What kind of error message we get with this kind of files:

In [9]:

i3 = dataio.I3File("/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1080.i3.zst"
)

print("opened")
print("more:", i3.more())


opened
more: False


In [10]:

frame = i3.pop_frame()



FATAL (dataio): no frame to pop (I3File.cxx:146 in I3FramePtr dataio::I3File::pop_frame(I3Frame::Stream))


RuntimeError: no frame to pop (in I3FramePtr dataio::I3File::pop_frame(I3Frame::Stream))

so these files should not be taken into account in anything. I added these files to paths.py

In [11]:
electron_log.loc[
    electron_log["i3_file_opened_ok"] == False,
    "batch_id"].isin(electron_weights["RunID"])


188     False
805     False
855     False
2594    False
3759    False
4543    False
4676    False
6631    False
Name: batch_id, dtype: bool

these problematic files are not in my event weight calculations, as it should not be.

In [12]:
electron_log = electron_log[electron_log["i3_file_opened_ok"] != False]


#### lets check if there are any other files that has no any event whose weight is calculated

In [13]:
electron_log[electron_log["weights_calculated"] == 0]


,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s


there is no. nice.

## Muon

In [14]:
len(muon_log[muon_log["i3_file_opened_ok"] == False])


5

In [15]:
muon_log[muon_log["i3_file_opened_ok"] == False]


,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s
4957,546,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Generator/genCard_546.lic,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_546.i3,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.084228
5711,616,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Generator/genCard_616.lic,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_616.i3,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.239421
5844,6282,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Generator/genCard_6282.lic,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_6282.i3,not ok,i3 file problem: read error while reading EventProperties at event 1. 0 EventProperties entries were read before the problem. Error: input stream error,False,0,0.022771
6654,706,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Generator/genCard_706.lic,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_706.i3,not ok,i3 file problem: read error while reading EventProperties at event 1. 0 EventProperties entries were read before the problem. Error: input stream error,False,0,0.118723
7553,7930,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Generator/genCard_7930.lic,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_7930.i3,not ok,i3 file problem: read error while reading EventProperties at event 1. 0 EventProperties entries were read before the problem. Error: input stream error,False,0,0.181714


### first idea is to check if these files are completely empthy. 

In [16]:
bad_muons = muon_log.loc[
    muon_log["i3_file_opened_ok"] == False,
    "i3_file"
]

In [17]:
for f in bad_muons:
    i3 = None
    try:
        i3 = dataio.I3File(f)
        stops = []

        while i3.more():
            frame = i3.pop_frame()
            stops.append(frame.Stop)

        print("READ OK:", f, stops)

    except RuntimeError as e:
        print("READ ERROR:", f, e)

    finally:
        if i3 is not None:
            i3.close()


READ OK: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_546.i3 [icetray.I3Frame.TrayInfo]
READ OK: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_616.i3 [icetray.I3Frame.TrayInfo]
READ ERROR: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_6282.i3 input stream error
READ ERROR: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_706.i3 input stream error
READ ERROR: /project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_7930.i3 input stream error


In [18]:

datasets = {
    "bad_muons": bad_muons}

rows = []

for dataset_name, files in datasets.items():
    for f in files:
        i3 = None
        counts = Counter()
        status = "OK"
        error = ""

        try:
            i3 = dataio.I3File(f)

            while i3.more():
                frame = i3.pop_frame()
                frame_type = str(frame.Stop).split(".")[-1]
                counts[frame_type] += 1

        except Exception as e:
            status = "ERROR"
            error = str(e)

        finally:
            if i3 is not None:
                i3.close()

        row = {
            "file": f,
            "status": status,
            "error": error,
        }
        row.update(counts)
        rows.append(row)

frame_counts_df = pd.DataFrame(rows).fillna(0)

# frame count columnlarını integer yap
meta_cols = ["dataset", "file", "status", "error"]
frame_cols = [c for c in frame_counts_df.columns if c not in meta_cols]
frame_counts_df[frame_cols] = frame_counts_df[frame_cols].astype(int)

frame_counts_df


# status here is different than status in muon_log

,file,status,error,TrayInfo
0,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_546.i3,OK,,1
1,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_616.i3,OK,,1
2,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_6282.i3,ERROR,input stream error,0
3,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_706.i3,ERROR,input stream error,1
4,/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_7930.i3,ERROR,input stream error,0


Interpretation of frame_counts_df:

- status: OK, error: NA/empty, TrayInfo: 1
  The file was opened and read successfully. It contains only one readable frame, and that frame is a TrayInfo frame. No DAQ/Physics/event frames were found.

- status: ERROR, error: "input stream error", TrayInfo: 0
  The file could be opened by I3File, but reading the first frame failed. No readable frames were recovered. This usually indicates a corrupted, truncated, or otherwise invalid I3 stream.

- status: ERROR, error: "input stream error", TrayInfo: 1
  The file was opened and at least one TrayInfo frame was read successfully, but reading a later frame failed. This suggests the file is partially readable but the stream becomes corrupted or incomplete after the TrayInfo frame.


so these files should not be taken into account in anything. I added these files to paths.py

In [19]:
muon_log.loc[
    muon_log["i3_file_opened_ok"] == False,
    "batch_id"].isin(muon_weights["RunID"])

4957    False
5711    False
5844    False
6654    False
7553    False
Name: batch_id, dtype: bool

these problematic files are not in my event weight calculations, as it should not be.

In [20]:
muon_log = muon_log[muon_log["i3_file_opened_ok"] != False]

lets check if there are any other files that has no any event whose weight is calculated

In [21]:
muon_log[muon_log["weights_calculated"] == 0]

,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s


#### there is no. nice.

## Tau 

In [22]:
len(tau_log[tau_log["i3_file_opened_ok"] == False])


1

In [23]:
tau_log[tau_log["i3_file_opened_ok"] == False]


,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s
3342,3950,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_3950.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_3950.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,1.23498


In [24]:
bad_tau = tau_log.loc[
    tau_log["i3_file_opened_ok"] == False,
    "i3_file"
]

#### first idea is to check if these files are completely empthy.

In [25]:
bad_tau = tau_log.loc[
    tau_log["i3_file_opened_ok"] == False,
    "i3_file"
]

In [26]:
# check if there is any kind of frame:

for f in bad_tau:
    i3 = dataio.I3File(f)

    if not i3.more():
        print("EMPTY:", f)
        i3.close()
        continue

    frame = i3.pop_frame()
    print("READ OK:", f, frame.Stop)

    i3.close()

EMPTY: /project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_3950.i3.zst


#### What kind of error message we get with this kind of files:

In [27]:
i3 = dataio.I3File("/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_3950.i3.zst"
)

print("opened")
print("more:", i3.more())

opened
more: False


In [28]:
frame = i3.pop_frame()

FATAL (dataio): no frame to pop (I3File.cxx:146 in I3FramePtr dataio::I3File::pop_frame(I3Frame::Stream))


RuntimeError: no frame to pop (in I3FramePtr dataio::I3File::pop_frame(I3Frame::Stream))

so this files should not be taken into account in anything. I added these files to paths.py

In [29]:
tau_log.loc[
    tau_log["i3_file_opened_ok"] == False,
    "batch_id"].isin(tau_weights["RunID"])

3342    False
Name: batch_id, dtype: bool

this problematic files are not in my event weight calculations, as it should not be.

In [30]:
tau_log = tau_log[tau_log["i3_file_opened_ok"] != False]

##### lets check if there are any other files that has no any event whose weight is calculated

In [ ]:
tau_log[tau_log["weights_calculated"] == 0]
## wtf??????

,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s
2,2,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_002.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_002.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,0.622973
4,4,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_004.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_004.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,0.563813
5,5,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_005.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_005.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,1.393014
8,8,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_008.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_008.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,1.169254
9,9,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_009.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_009.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,0.610393
11,11,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_011.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_011.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,0.982062
12,12,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_012.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_012.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,0.563346
1288,208,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_208.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_208.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,1.271524
1464,224,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_224.lic,/project/6008051/pone_simulation/MC000004-nu_tau-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_224.i3.zst,not ok,weighting problem at event 1: oneweight could not be computed. 0 event weights were calculated before the problem. Error: Out of declared generation phase space. Impossible event.,True,0,3.312323
2586,326,/

In [ ]:
len(tau_log[tau_log["weights_calculated"] == 0])
 
#  I NEED TO LOOK HERE IN MORE DETAIL!


13

## NC

In [33]:
len(nc_log[nc_log["i3_file_opened_ok"] == False])


2

In [34]:
nc_log[nc_log["i3_file_opened_ok"] == False]


,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s
1569,2335,/project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/genCard_2335.lic,/project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/gen_2335.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.263894
8961,9055,/project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/genCard_9055.lic,/project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/gen_9055.i3.zst,not ok,i3 file problem: no DAQ frames with EventProperties and I3EventHeader were found in the EventProperties i3 file.,False,0,0.939840


### first idea is to check if these files are completely empthy.

In [35]:
bad_nc = nc_log.loc[
    nc_log["i3_file_opened_ok"] == False,
    "i3_file"
]

In [36]:
# check if there is any kind of frame:

for f in bad_nc:
    i3 = dataio.I3File(f)

    if not i3.more():
        print("EMPTY:", f)
        i3.close()
        continue

    frame = i3.pop_frame()
    print("READ OK:", f, frame.Stop)

    i3.close()

EMPTY: /project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/gen_2335.i3.zst
EMPTY: /project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/gen_9055.i3.zst


#### What kind of error message we get with this kind of files:

In [37]:
i3 = dataio.I3File("/project/6008051/pone_simulation/MC000005-nu_NC-2_7-LeptonInjector_PROPOSAL_clsim_NC-v10/Generator/gen_2335.i3.zst")

print("opened")
print("more:", i3.more())

opened
more: False


In [38]:
frame = i3.pop_frame()

FATAL (dataio): no frame to pop (I3File.cxx:146 in I3FramePtr dataio::I3File::pop_frame(I3Frame::Stream))


RuntimeError: no frame to pop (in I3FramePtr dataio::I3File::pop_frame(I3Frame::Stream))

##### so these files should not be taken into account in anything. I added these files to paths.py

In [39]:
nc_log.loc[
    nc_log["i3_file_opened_ok"] == False,
    "batch_id"].isin(nc_weights["RunID"])

1569    False
8961    False
Name: batch_id, dtype: bool

these problematic files are not in my event weight calculations, as it should not be.

In [40]:
nc_log = nc_log[nc_log["i3_file_opened_ok"] != False]

lets check if there are any other files that has no any event whose weight is calculated

In [41]:
nc_log[nc_log["weights_calculated"] == 0]

,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s


there is no. nice.

# I3 Files That Has < 200 & > 0 Available Events

In [42]:
electron_log[
    (electron_log["weights_calculated"] > 0) &
    (electron_log["weights_calculated"] < 200)
]


,batch_id,lic_file,i3_file,status,error,i3_file_opened_ok,weights_calculated,duration_s
265,1150,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_1150.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1150.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 166. 165 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 166: read error. 165 event weights were calculated before the problem. Error: input stream error,True,165,0.507415
641,1492,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_1492.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1492.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 123. 122 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 123: read error. 122 event weights were calculated before the problem. Error: input stream error,True,122,0.951075
1599,2363,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_2363.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2363.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 100. 99 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 100: read error. 99 event weights were calculated before the problem. Error: input stream error,True,99,1.290242
1644,2404,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_2404.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2404.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 80. 79 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 80: read error. 79 event weights were calculated before the problem. Error: input stream error,True,79,2.907490
1659,2418,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_2418.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2418.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 118. 117 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 118: read error. 117 event weights were calculated before the problem. Error: input stream error,True,117,0.686976
2077,2798,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_2798.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2798.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 123. 122 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 123: read error. 122 event weights were calculated before the problem. Error: input stream error,True,122,2.572251
2230,2959,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_2959.lic,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2959.i3.zst,partially ok,i3 file problem: read error while reading EventProperties at event 159. 158 EventProperties entries were read before the problem. Error: input stream error | i3 file problem at event 159: read error. 158 event weights were calculated before the problem. Error: input stream error,True,158,0.555234
2344,3157,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/genCard_3

In [44]:
rows = []

for i3_file in electron_log[
    (electron_log["weights_calculated"] > 0) &
    (electron_log["weights_calculated"] < 200)
]["i3_file"]:
    counts = Counter()
    error = ""

    try:
        f = dataio.I3File(str(i3_file))

        while f.more():
            try:
                frame = f.pop_frame()
            except Exception as e:
                error = str(e)
                break

            counts[str(frame.Stop)] += 1

        f.close()

    except Exception as e:
        error = str(e)

    rows.append({
        "i3_file": i3_file,
        "DAQ": counts[str(icetray.I3Frame.DAQ)],
        "TrayInfo": counts[str(icetray.I3Frame.TrayInfo)],
        "Geometry": counts[str(icetray.I3Frame.Geometry)],
        "Calibration": counts[str(icetray.I3Frame.Calibration)],
        "DetectorStatus": counts[str(icetray.I3Frame.DetectorStatus)],
        "Physics": counts[str(icetray.I3Frame.Physics)],
        "error": error,
    })

frame_counts = pd.DataFrame(rows)
frame_counts


,i3_file,DAQ,TrayInfo,Geometry,Calibration,DetectorStatus,Physics,error
0,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1150.i3.zst,165,1,0,0,0,0,input stream error
1,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_1492.i3.zst,122,1,0,0,0,0,input stream error
2,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2363.i3.zst,99,1,0,0,0,0,input stream error
3,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2404.i3.zst,79,1,0,0,0,0,input stream error
4,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2418.i3.zst,117,1,0,0,0,0,input stream error
5,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2798.i3.zst,122,1,0,0,0,0,input stream error
6,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_2959.i3.zst,158,1,0,0,0,0,input stream error
7,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_3157.i3.zst,30,1,0,0,0,0,input stream error
8,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_3491.i3.zst,169,1,0,0,0,0,input stream error
9,/project/6008051/pone_simulation/MC000003-nu_e-2_7-LeptonInjector_PROPOSAL_clsim-v10/Generator/gen_3882.i3.zst,174,1,0,0,0,0,input stream error


so oll of them has "input stream error" which means events after some point are not reachable.

In [ ]:
muon_log[
    (muon_log["weights_calculated"] > 0) &
    (muon_log["weights_calculated"] < 200)
]


In [ ]:
tau_log[
    (tau_log["weights_calculated"] > 0) &
    (tau_log["weights_calculated"] < 200)
]


In [ ]:
nc_log[
    (nc_log["weights_calculated"] > 0) &
    (nc_log["weights_calculated"] < 200)
]
